# Tree comparison

Run once by the `tree_comparison` rule after all trees are built. Compares the `{derep}` x
`{copies}` trees (and, if a subset was used, only the subset trees): topological distance,
fragmentation and a single- vs multi-copy comparison of how well each gene separates clades.

In [ ]:
import json, os, sys
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
%matplotlib inline
from Bio import Phylo

P = json.load(open(os.environ["NB_PARAMS"]))
sys.path.insert(0, P["scripts_dir"])
import tree_accuracy_lib as lib
import tree_diagnostics_lib as dlib
OUT = Path(P["out_dir"]); OUT.mkdir(parents=True, exist_ok=True)
taxonomy = lib.read_taxonomy(P["taxonomy"])
trees = {t["label"]: t for t in P["trees"] if Path(t["tree"]).exists()}
print("trees found:", list(trees))
loaded = {k: Phylo.read(v["tree"], "newick") for k, v in trees.items()}
for k, t in loaded.items():
    print(f"  {k}: {t.count_terminals()} tips")

## Topological distance (normalised Robinson-Foulds on shared tips)

In [ ]:
labels = list(loaded)
rf = np.full((len(labels), len(labels)), np.nan)
for i, a in enumerate(labels):
    for j, b in enumerate(labels):
        rf[i, j] = 0.0 if i == j else lib.rf_distance(loaded[a], loaded[b])[1]
print("\t" + "\t".join(labels))
for i, a in enumerate(labels):
    print(a + "\t" + "\t".join(f"{x:.3f}" for x in rf[i]))
with open(OUT / "rf_matrix.tsv", "w") as f:
    f.write("tree\t" + "\t".join(labels) + "\n")
    for i, a in enumerate(labels):
        f.write(a + "\t" + "\t".join(f"{x:.4f}" for x in rf[i]) + "\n")
if len(labels) > 1:
    fig, ax = plt.subplots(figsize=(5, 4)); im = ax.imshow(rf, vmin=0); ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels); plt.colorbar(im); plt.show()

## Fragmentation by tree

In [ ]:
table = {}
names_by_tree = {}
for k, t in loaded.items():
    s, n = lib.fragmentation(t, taxonomy)
    table[k] = {r["rank"]: r for r in s}
    names_by_tree[k] = {(x["rank"], x["name"]): x["pieces"] for x in n}
print(f"{'tree':22}" + "".join(f"{r:>20}" for r in lib.RANKS))
for k in labels:
    print(f"{k:22}" + "".join(f"{table[k][r]['extra_pieces']:>9} extra/{table[k][r]['fragmented_names']:>3} nm" for r in lib.RANKS))
with open(OUT / "fragmentation_by_tree.tsv", "w") as f:
    f.write("tree\trank\tnames\tfragmented_names\textra_pieces\ttips_in_fragmented\n")
    for k in labels:
        for r in lib.RANKS:
            x = table[k][r]; f.write(f"{k}\t{r}\t{x['names']}\t{x['fragmented_names']}\t{x['extra_pieces']}\t{x['tips_in_fragmented']}\n")
# names whose piece count differs between trees
keys = sorted(set().union(*[set(v) for v in names_by_tree.values()])) if names_by_tree else []
diff = [(k, [names_by_tree[t].get(k, 0) for t in labels]) for k in keys if len({names_by_tree[t].get(k, 0) for t in labels}) > 1]
print(f"\n{len(diff)} names have a different piece count in at least one tree; largest differences:")
for (rank, name), v in sorted(diff, key=lambda d: -(max(d[1]) - min(d[1])))[:15]:
    print(f"  {rank:7} {name:28}", dict(zip(labels, v)))

## Single vs multi copy

For each `{derep}`, does adding the top copy of Duplicated genes make genes separate clades better or
worse? `separation` is the chance that a within-clade pair is closer than a between-clade pair
(1 = perfect). Same genomes are compared in both alignments.

In [ ]:
import random
for derep in sorted({v["derep"] for v in trees.values()}):
    s, m = trees.get(f"{derep}/single"), trees.get(f"{derep}/multi")
    if not s or not m or not (Path(s["alignment"]).exists() and Path(m["alignment"]).exists()):
        print(derep, ": need both single and multi alignments"); continue
    results = {}
    for kind, t in (("single", s), ("multi", m)):
        ids, matrix = dlib.load_alignment(t["alignment"])
        parts = dlib.read_partitions(t["partitions"])
        _, present = dlib.occupancy(ids, matrix, parts)
        rep = dlib.rank_report(ids, matrix, present, parts, taxonomy, ranks=["order", "family"], n_pairs=P["n_pairs"], seed=3)
        results[kind] = {rank: {g: sep for g, sep, n in dlib.gene_separation(rep, parts, rank)} for rank in rep}
        results[kind]["summary"] = {rank: rep[rank]["summary"] for rank in rep}
    print(f"\n== {derep}")
    for rank in ("order", "family"):
        a, b = results["single"][rank], results["multi"][rank]
        common = [g for g in a if g in b and not (np.isnan(a[g]) or np.isnan(b[g]))]
        d = np.array([b[g] - a[g] for g in common])
        print(f"{rank:7} genes {len(common)}  median separation single {np.median([a[g] for g in common]):.3f} "
              f"multi {np.median([b[g] for g in common]):.3f}  multi better in {(d > 0).mean():.0%}  "
              f"overall pair separation single {results['single']['summary'][rank]['separation_pdist']:.3f} "
              f"multi {results['multi']['summary'][rank]['separation_pdist']:.3f}")
        with open(OUT / f"gene_separation_{derep}_{rank}.tsv", "w") as f:
            f.write("gene\tsingle\tmulti\n")
            for g in common:
                f.write(f"{g}\t{a[g]:.4f}\t{b[g]:.4f}\n")